# ✈️ Flight Delay Prediction Pipeline

This notebook implements an end-to-end, production-ready machine learning pipeline to predict flight delays (>= 15 minutes). 

### Key Features:
- **Zero Data Leakage:** All preprocessing, transformations, and target encoding are isolated within Scikit-Learn `Pipeline`s.
- **Advanced Feature Engineering:** Calculates Haversine geospatial distances, time-based cyclic encodings, and airport traffic load.
- **Robust Cross-Validation:** Uses `StratifiedKFold` to handle severe class imbalances and safely computes `ROC-AUC`.
- **Threshold Optimization:** Optimizes prediction thresholds automatically using custom objective equations.


In [ ]:
import os
import pandas as pd
import numpy as np
import logging
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve, roc_curve
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.cluster import KMeans
from sklearn.base import BaseEstimator, TransformerMixin, ClassifierMixin
from sklearn.calibration import CalibratedClassifierCV
import joblib
import warnings

warnings.filterwarnings('ignore')

# --- DIRECTORY SETUP ---
OUTPUT_DIRS = ['output/plots', 'output/models', 'output/logs', 'output/metrics', 'output/reports']
for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)


## 1. Custom Scikit-Learn Transformers
We define robust classes to handle cyclic time features, geospatial distances, traffic load, and Out-Of-Fold (OOF) target encodings safely inside our pipeline.


In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth radius in miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

class FlightFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.origin_traffic_ = {}
        self.dest_traffic_ = {}
        self.route_freq_ = {}
        self.major_cities = ['New York', 'Los Angeles', 'Chicago', 'Atlanta', 'Dallas-Fort Worth', 'Dallas', 'Houston', 'Denver']
        
    def fit(self, X, y=None):
        self.origin_traffic_ = X['ORIGIN_AIRPORT'].value_counts().to_dict()
        self.dest_traffic_ = X['DESTINATION_AIRPORT'].value_counts().to_dict()
        
        route_str = X['ORIGIN_AIRPORT'].astype(str) + '_' + X['DESTINATION_AIRPORT'].astype(str)
        self.route_freq_ = route_str.value_counts().to_dict()
        return self

    def transform(self, X, y=None):
        X_out = X.copy()
        try:
            dep_str = X_out['SCHEDULED_DEPARTURE'].astype(float).astype(int).astype(str).str.zfill(4)
            X_out['HOUR'] = dep_str.str[:2].astype(float)
        except Exception:
            X_out['HOUR'] = 0.0
        X_out['HOUR'] = X_out['HOUR'].fillna(0)
        
        # Cyclic encoding
        X_out['MONTH'] = X_out['MONTH'].fillna(1)
        X_out['DAY'] = X_out['DAY'].fillna(1)
        X_out['DAY_OF_WEEK'] = X_out['DAY_OF_WEEK'].fillna(1)

        X_out['MONTH_sin'] = np.sin(2 * np.pi * X_out['MONTH'] / 12.0)
        X_out['MONTH_cos'] = np.cos(2 * np.pi * X_out['MONTH'] / 12.0)
        X_out['DAY_sin'] = np.sin(2 * np.pi * X_out['DAY'] / 31.0)
        X_out['DAY_cos'] = np.cos(2 * np.pi * X_out['DAY'] / 31.0)
        X_out['DAY_OF_WEEK_sin'] = np.sin(2 * np.pi * X_out['DAY_OF_WEEK'] / 7.0)
        X_out['DAY_OF_WEEK_cos'] = np.cos(2 * np.pi * X_out['DAY_OF_WEEK'] / 7.0)
        X_out['HOUR_sin'] = np.sin(2 * np.pi * X_out['HOUR'] / 24.0)
        X_out['HOUR_cos'] = np.cos(2 * np.pi * X_out['HOUR'] / 24.0)
        
        # Flags
        X_out['is_weekend'] = (X_out['DAY_OF_WEEK'] > 5).astype(int)
        X_out['is_peak_hour'] = (((X_out['HOUR'] >= 7) & (X_out['HOUR'] <= 9)) | ((X_out['HOUR'] >= 16) & (X_out['HOUR'] <= 19))).astype(int)
        X_out['is_night_flight'] = ((X_out['HOUR'] >= 22) | (X_out['HOUR'] <= 5)).astype(int)
        
        # Speed
        time_valid = X_out['SCHEDULED_TIME'].replace(0, np.nan)
        X_out['SPEED'] = X_out['DISTANCE'] / time_valid
        
        # Advanced Features
        X_out['ROUTE'] = X_out['ORIGIN_AIRPORT'].astype(str) + '_' + X_out['DESTINATION_AIRPORT'].astype(str)
        X_out['GEO_DISTANCE'] = haversine(
            X_out['ORIGIN_LATITUDE'].fillna(0), X_out['ORIGIN_LONGITUDE'].fillna(0), 
            X_out['DEST_LATITUDE'].fillna(0), X_out['DEST_LONGITUDE'].fillna(0)
        )
        
        X_out['SAME_STATE'] = (X_out['ORIGIN_STATE'] == X_out['DEST_STATE']).astype(int)
        X_out['MAJOR_CITY'] = (X_out['ORIGIN_CITY'].isin(self.major_cities) | X_out['DEST_CITY'].isin(self.major_cities)).astype(int)
        
        orig_t = X_out['ORIGIN_AIRPORT'].map(self.origin_traffic_).fillna(1)
        dest_t = X_out['DESTINATION_AIRPORT'].map(self.dest_traffic_).fillna(1)
        X_out['AIRPORT_TRAFFIC'] = orig_t + dest_t
        X_out['AIRPORT_IMPORTANCE'] = (orig_t * dest_t) / 1000.0  
        X_out['ROUTE_FREQUENCY'] = X_out['ROUTE'].map(self.route_freq_).fillna(1)
        X_out['DISTANCE_CATEGORY'] = pd.cut(X_out['DISTANCE'], bins=[0, 500, 1500, 10000], labels=[1, 2, 3]).astype(float).fillna(2)
        X_out['LONG_HAUL'] = (X_out['GEO_DISTANCE'] > 2000).astype(int)
        
        # Interaction features
        X_out['TIME_DISTANCE'] = X_out['HOUR'] * X_out['DISTANCE']
        X_out['TRAFFIC_PRESSURE'] = X_out['AIRPORT_TRAFFIC'] * X_out['is_peak_hour']
        X_out['ROUTE_COMPLEXITY'] = X_out['ROUTE_FREQUENCY'] / (X_out['DISTANCE'] + 1)
        
        cols_to_drop = ['MONTH', 'DAY', 'DAY_OF_WEEK', 'HOUR', 'SCHEDULED_DEPARTURE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_ARRIVAL', 'ORIGIN_LATITUDE', 'ORIGIN_LONGITUDE', 'DEST_LATITUDE', 'DEST_LONGITUDE', 'ORIGIN_CITY', 'ORIGIN_STATE', 'DEST_CITY', 'DEST_STATE', 'AIRLINE_NAME', 'AIRLINE']
        return X_out.drop(columns=[c for c in cols_to_drop if c in X_out.columns])

class OOFTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols, cv=5, alpha=10):
        self.cols = cols
        self.cv = cv
        self.alpha = alpha

    def fit(self, X, y):
        y_series = pd.Series(y, index=X.index)
        self.global_means_ = {}
        self.category_means_ = {}
        
        for col in self.cols:
            self.global_means_[col] = y_series.mean()
            stats = y_series.groupby(X[col]).agg(['mean', 'count'])
            self.category_means_[col] = ((stats['count'] * stats['mean'] + self.alpha * self.global_means_[col]) / (stats['count'] + self.alpha)).to_dict()
        return self

    def transform(self, X, y=None):
        X_out = X.copy()
        if y is not None:
            y_series = pd.Series(y, index=X.index)
            kf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)
            for col in self.cols:
                oof_encoded = pd.Series(np.nan, index=X.index)
                for train_idx, val_idx in kf.split(X, y):
                    train_x, val_x = X.iloc[train_idx], X.iloc[val_idx]
                    train_y = y_series.iloc[train_idx]
                    stats = train_y.groupby(train_x[col]).agg(['mean', 'count'])
                    smooth_mean = (stats['count'] * stats['mean'] + self.alpha * self.global_means_[col]) / (stats['count'] + self.alpha)
                    oof_encoded.iloc[val_idx] = val_x[col].map(smooth_mean).fillna(self.global_means_[col])
                X_out[col + '_TE'] = oof_encoded
                X_out.drop(columns=[col], inplace=True)
        else:
            for col in self.cols:
                X_out[col + '_TE'] = X[col].map(self.category_means_[col]).fillna(self.global_means_[col])
                X_out.drop(columns=[col], inplace=True)
        return X_out

class FullPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.engineer = FlightFeatureEngineer()
        self.imputer_num = SimpleImputer(strategy='median')
        self.imputer_cat = SimpleImputer(strategy='constant', fill_value='Unknown')
        self.te = OOFTargetEncoder(cols=['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT'], alpha=10)
        self.scaler = StandardScaler()
        
    def fit(self, X, y):
        X_temp = X.copy()
        X_temp['ROUTE'] = X_temp['ORIGIN_AIRPORT'].astype(str) + '_' + X_temp['DESTINATION_AIRPORT'].astype(str)
        cat_cols = ['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
        self.imputer_cat.fit(X_temp[cat_cols])
        
        cat_imputed = pd.DataFrame(self.imputer_cat.transform(X_temp[cat_cols]), columns=cat_cols, index=X.index)
        self.te.fit(cat_imputed, y)
        
        X_eng = self.engineer.fit_transform(X)
        num_cols = [c for c in X_eng.columns if c not in cat_cols and c != 'ROUTE']
        self.imputer_num.fit(X_eng[num_cols])
        
        X_combined = pd.concat([pd.DataFrame(self.imputer_num.transform(X_eng[num_cols]), columns=num_cols, index=X.index), self.te.transform(cat_imputed, y)], axis=1)
        self.scaler.fit(X_combined)
        self.final_cols = X_combined.columns
        return self
        
    def transform(self, X, y=None):
        X_temp = X.copy()
        X_temp['ROUTE'] = X_temp['ORIGIN_AIRPORT'].astype(str) + '_' + X_temp['DESTINATION_AIRPORT'].astype(str)
        cat_cols = ['AIRLINE', 'ROUTE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT']
        cat_imputed = pd.DataFrame(self.imputer_cat.transform(X_temp[cat_cols]), columns=cat_cols, index=X.index)
        
        X_eng = self.engineer.transform(X)
        num_cols = [c for c in X_eng.columns if c not in cat_cols and c != 'ROUTE']
        
        X_combined = pd.concat([pd.DataFrame(self.imputer_num.transform(X_eng[num_cols]), columns=num_cols, index=X.index), self.te.transform(cat_imputed, y)], axis=1)
        return pd.DataFrame(self.scaler.transform(X_combined), columns=X_combined.columns, index=X.index)

class WeightedAdaBoost(BaseEstimator, ClassifierMixin):
    def __init__(self, estimator=None, n_estimators=400, learning_rate=0.03, random_state=42):
        if estimator is None:
            estimator = DecisionTreeClassifier(max_depth=4, min_samples_leaf=20)
        self.estimator = estimator
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.random_state = random_state
        self.model = AdaBoostClassifier(estimator=self.estimator, n_estimators=self.n_estimators, learning_rate=self.learning_rate, random_state=self.random_state)

    def fit(self, X, y):
        self.model.fit(X, y, sample_weight=np.where(y == 1, 2.0, 1.0))
        self.classes_ = self.model.classes_
        return self

    def predict(self, X): return self.model.predict(X)
    def predict_proba(self, X): return self.model.predict_proba(X)
    @property
    def feature_importances_(self): return self.model.feature_importances_


## 2. Safe Data Loading & Merging
We load flight data and safely attach `airports` and `airlines` context mappings without causing target leakage.


In [ ]:
# Load Datasets
flights = pd.read_csv('flights.csv', usecols=[
    'MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 
    'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'SCHEDULED_ARRIVAL', 
    'SCHEDULED_TIME', 'DISTANCE', 'ARRIVAL_DELAY'
])
airlines = pd.read_csv('airlines.csv')
airports = pd.read_csv('airports.csv')

flights = flights.dropna(subset=['ARRIVAL_DELAY'])
flights['DELAYED'] = (flights['ARRIVAL_DELAY'] >= 15).astype(int)

# Merge Airlines
flights = flights.merge(airlines, left_on='AIRLINE', right_on='IATA_CODE', how='left')
flights.rename(columns={'AIRLINE_y': 'AIRLINE_NAME', 'AIRLINE_x': 'AIRLINE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

# Merge Origin Airports
flights = flights.merge(airports[['IATA_CODE', 'CITY', 'STATE', 'LATITUDE', 'LONGITUDE']], 
                        left_on='ORIGIN_AIRPORT', right_on='IATA_CODE', how='left')
flights.rename(columns={'CITY': 'ORIGIN_CITY', 'STATE': 'ORIGIN_STATE', 'LATITUDE': 'ORIGIN_LATITUDE', 'LONGITUDE': 'ORIGIN_LONGITUDE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

# Merge Dest Airports
flights = flights.merge(airports[['IATA_CODE', 'CITY', 'STATE', 'LATITUDE', 'LONGITUDE']], 
                        left_on='DESTINATION_AIRPORT', right_on='IATA_CODE', how='left')
flights.rename(columns={'CITY': 'DEST_CITY', 'STATE': 'DEST_STATE', 'LATITUDE': 'DEST_LATITUDE', 'LONGITUDE': 'DEST_LONGITUDE'}, inplace=True)
if 'IATA_CODE' in flights.columns: flights.drop(columns=['IATA_CODE'], inplace=True)

X_full = flights.drop(columns=['ARRIVAL_DELAY', 'DELAYED'])
y_full = flights['DELAYED']


## 3. Subsampling & Preprocessing
To manage memory and compute times, we subsample the data and process it strictly separated from the test-set.


In [ ]:
SAMPLE_SIZE = 50000 
if len(flights) > SAMPLE_SIZE:
    _, X_full, _, y_full = train_test_split(X_full, y_full, test_size=SAMPLE_SIZE, stratify=y_full, random_state=42)

# Global Test Split (never touched until final validation)
X_train, X_test, y_train, y_test = train_test_split(X_full, y_full, test_size=0.2, stratify=y_full, random_state=42)
X_test_original = X_test.copy()

# Preprocess
preprocessor = FullPreprocessor()
X_train_preprocessed = preprocessor.fit_transform(X_train, y_train)
X_train_preprocessed.head()


## 4. Forward Feature Selection
Extracting the top 22 most highly correlated engineered features via logistic sequential testing.


In [ ]:
lr_evaluator = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
sfs = SequentialFeatureSelector(lr_evaluator, n_features_to_select=22, direction='forward', cv=5, n_jobs=-1)

sfs.fit(X_train_preprocessed, y_train)
selected_features = X_train_preprocessed.columns[sfs.get_support()].tolist()

print(f"Selected Features ({len(selected_features)}): {selected_features}")
X_train_sel = X_train_preprocessed[selected_features]


## 5. Model Training & Cross-Validation
Evaluating models securely. A strict manual `StratifiedKFold(n_splits=3)` ensures `NaN` values from highly-imbalanced folds do not disrupt the pipeline evaluation logic.


In [ ]:
models = {
    'Logistic Regression': Pipeline([('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))]),
    'Polynomial Regression': Pipeline([
        ('poly', PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)),
        ('model', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
    ]),
    'AdaBoost': Pipeline([
        ('model', WeightedAdaBoost(estimator=DecisionTreeClassifier(max_depth=4, min_samples_leaf=20), n_estimators=400, learning_rate=0.03))
    ])
}

model_results = {}
best_f1 = 0
best_model_name = ""

kf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for name, pipeline in models.items():
    print(f"--- Cross-Validating {name} ---")
    fold_f1s = []
    fold_aucs = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_sel, y_train)):
        X_fold_train, X_fold_val = X_train_sel.iloc[train_idx], X_train_sel.iloc[val_idx]
        y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        
        unique_classes, counts = np.unique(y_fold_val, return_counts=True)
        if len(unique_classes) < 2:
            print(f"[{name}] Fold {fold+1} contains only one class -> AUC undefined!")
            
        pipeline.fit(X_fold_train, y_fold_train)
        try:
            y_pred = pipeline.predict(X_fold_val)
            y_proba = pipeline.predict_proba(X_fold_val)[:, 1]
            
            auc = np.nan if len(unique_classes) < 2 else roc_auc_score(y_fold_val, y_proba)
            f1 = f1_score(y_fold_val, y_pred)
            
            fold_f1s.append(f1)
            fold_aucs.append(auc)
        except Exception as e:
            print(f"[{name}] Fold {fold+1} prediction failed: {str(e)}")
            raise e 
            
    mean_f1 = np.nanmean(fold_f1s)
    mean_auc = np.nanmean(fold_aucs)
    
    model_results[name] = {'F1 (CV)': mean_f1, 'ROC-AUC (CV)': mean_auc}
    print(f"{name} -> Mean CV F1: {mean_f1:.4f}, Mean CV AUC: {mean_auc:.4f}")
    
    pipeline.fit(X_train_sel, y_train)
    if mean_f1 > best_f1:
        best_f1 = mean_f1
        best_model_name = name

# Display Summary
df_results = pd.DataFrame(model_results).T
display(df_results)


## 6. Threshold Optimization & Probability Calibration
Fine-tuning probabilities into classifications based strictly on maximizing our objective equation: `Score = (0.7 * F1) + (0.5 * Recall)`.


In [ ]:
best_model_base = models[best_model_name]
calibrated_model = CalibratedClassifierCV(best_model_base, method='sigmoid', cv=3)
calibrated_model.fit(X_train_sel, y_train)

y_probs_cv = cross_val_predict(best_model_base, X_train_sel, y_train, cv=3, method='predict_proba', n_jobs=-1)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_train, y_probs_cv)

f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-10)
optim_scores = (0.7 * f1_scores) + (0.5 * recalls[:-1])

valid_idx = np.where((thresholds >= 0.1) & (thresholds <= 0.9))[0]
if len(valid_idx) > 0:
    best_idx = valid_idx[np.argmax(optim_scores[valid_idx])]
    best_threshold = thresholds[best_idx]
    print(f"Optimal Threshold: {best_threshold:.4f} (Objective Score: {optim_scores[best_idx]:.4f})")
else:
    best_threshold = 0.5


## 7. Final Test Evaluation & Outputs
Validating the optimized threshold rules against the completely isolated test split and visualizing the results.


In [ ]:
X_test_preprocessed = preprocessor.transform(X_test, y=None)
X_test_sel = X_test_preprocessed[selected_features]

test_probs = calibrated_model.predict_proba(X_test_sel)[:, 1]
test_preds = (test_probs >= best_threshold).astype(int)

print(f"Test Accuracy: {accuracy_score(y_test, test_preds):.4f}")
print(f"Test F1: {f1_score(y_test, test_preds):.4f}")
print(f"Test Recall: {recall_score(y_test, test_preds):.4f}")
print(f"Test AUC: {roc_auc_score(y_test, test_probs):.4f}")

cm = confusion_matrix(y_test, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Final Confusion Matrix')
plt.show()


## 8. Exporting Serialized Pipelines
We group all artifacts, clusters, and the calibrated model inside a `joblib` object for future inference via the Streamlit web app.


In [ ]:
kmeans = KMeans(n_clusters=6, random_state=42)
train_clusters = kmeans.fit_predict(X_train_sel)

final_pipeline_obj = {
    'preprocessor': preprocessor,
    'feature_selector': sfs,
    'selected_features': selected_features,
    'base_model': best_model_base,
    'calibrated_model': calibrated_model,
    'best_threshold': best_threshold,
    'cluster_model': kmeans
}

joblib.dump(final_pipeline_obj, 'output/models/model.joblib')
print("Pipeline Successfully Saved! 🚀")
